In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FILE_FORECAST = 'Hasil Forecast 2022-2025 (ARIMA & GB & Ensemble).xlsx'

df_komparasi = pd.read_excel(FILE_FORECAST, sheet_name='Komparasi_Model')
df_metrik    = pd.read_excel(FILE_FORECAST, sheet_name='Metrik_Per_Unit')
df_excl      = pd.read_excel(FILE_FORECAST, sheet_name='Unit_Dikecualikan')

print(f"Komparasi_Model  : {len(df_komparasi)} baris, {df_komparasi['EQUIP NAME'].nunique()} unit")
print(f"Metrik_Per_Unit  : {len(df_metrik)} unit")
print(f"Unit_Dikecualikan: {len(df_excl)} unit")

Komparasi_Model  : 2076 baris, 173 unit
Metrik_Per_Unit  : 173 unit
Unit_Dikecualikan: 66 unit


### F1: List Data Simpangan Tiap Unit yang MAPE > 35%

In [4]:
# Ambil unit dengan MAPE >= 35%
unit_over35 = df_metrik[df_metrik['MAPE_Terpilih (%)'] >= 35]['EQUIP NAME'].tolist()

# Ambil data bulanan dari Komparasi_Model untuk unit tersebut
df_detail = df_komparasi[df_komparasi['EQUIP NAME'].isin(unit_over35)].copy()

# Ambil kolom prediksi terpilih (sudah ada di Komparasi_Model)
df_detail_tampil = df_detail[[
    'EQUIP NAME', 'NAMA_MASTER_TERPETAKAN', 'Bulan',
    'Aktual_HM', 'Prediksi_HM_Terpilih',
    'Model_Terpilih_Unit', 'MAPE_Terpilih (%)',
    'RMSE_HM_ARIMA', 'RMSE_HM_GB', 'RMSE_HM_Ensemble'
]].copy()

# Tambah kolom RMSE terpilih sesuai model yang dipilih per unit
def ambil_rmse_terpilih(row):
    if row['Model_Terpilih_Unit'] == 'ARIMA':
        return row['RMSE_HM_ARIMA']
    elif row['Model_Terpilih_Unit'] == 'Gradient Boosting':
        return row['RMSE_HM_GB']
    else:
        return row['RMSE_HM_Ensemble']

df_detail_tampil['RMSE_Terpilih'] = df_detail_tampil.apply(ambil_rmse_terpilih, axis=1)

# Tambah kolom selisih absolut prediksi vs aktual per bulan
df_detail_tampil['Selisih_HM'] = (
    df_detail_tampil['Prediksi_HM_Terpilih'] - df_detail_tampil['Aktual_HM']
).round(2)

# Urutkan per unit (MAPE terbesar) lalu per bulan
df_mape_per_unit = df_metrik[['EQUIP NAME', 'MAPE_Terpilih (%)']].copy()
df_detail_tampil = df_detail_tampil.merge(
    df_mape_per_unit.rename(columns={'MAPE_Terpilih (%)': 'MAPE_Unit'}),
    on='EQUIP NAME', how='left'
)
df_detail_tampil = df_detail_tampil.sort_values(
    ['MAPE_Unit', 'EQUIP NAME', 'Bulan'],
    ascending=[False, True, True]
).drop(columns=['MAPE_Unit']).reset_index(drop=True)

# Ringkasan per unit
df_ringkasan_over35 = df_metrik[df_metrik['MAPE_Terpilih (%)'] >= 35][[
    'EQUIP NAME', 'NAMA_MASTER_TERPETAKAN',
    'Model_Terpilih_Unit', 'MAPE_Terpilih (%)',
    'RMSE_HM_ARIMA', 'RMSE_HM_GB', 'RMSE_HM_Ensemble'
]].copy()

def ambil_rmse_ringkasan(row):
    if row['Model_Terpilih_Unit'] == 'ARIMA':
        return row['RMSE_HM_ARIMA']
    elif row['Model_Terpilih_Unit'] == 'Gradient Boosting':
        return row['RMSE_HM_GB']
    else:
        return row['RMSE_HM_Ensemble']

df_ringkasan_over35['RMSE_Terpilih'] = df_ringkasan_over35.apply(ambil_rmse_ringkasan, axis=1)
df_ringkasan_over35 = df_ringkasan_over35.sort_values('MAPE_Terpilih (%)', ascending=False).reset_index(drop=True)

from IPython.display import display
print(f"=== ANALISA 1: Unit MAPE > 35% — Ringkasan Per Unit ===")
print(f"Total unit: {len(df_ringkasan_over35)}\n")
display(df_ringkasan_over35.round(2))

print(f"\n=== Detail Prediksi vs Aktual HM Per Bulan ===")
display(df_detail_tampil[[
    'EQUIP NAME', 'NAMA_MASTER_TERPETAKAN', 'Bulan',
    'Aktual_HM', 'Prediksi_HM_Terpilih', 'Selisih_HM',
    'Model_Terpilih_Unit', 'MAPE_Terpilih (%)', 'RMSE_Terpilih'
]].round(2))

=== ANALISA 1: Unit MAPE > 35% — Ringkasan Per Unit ===
Total unit: 70



,EQUIP NAME,NAMA_MASTER_TERPETAKAN,Model_Terpilih_Unit,MAPE_Terpilih (%),RMSE_HM_ARIMA,RMSE_HM_GB,RMSE_HM_Ensemble,RMSE_Terpilih
0,SAKURA,SAKURA,Gradient Boosting,1109.74,51.46,51.15,51.30,51.15
1,L 9103 NN EX. L 8781 LY,L 9103 NN EX. L 8781 LY,ARIMA,613.34,26.53,26.55,26.52,26.53
2,PINUS,PINUS,ARIMA,417.26,76.12,74.11,75.02,76.12
3,PALEM,PALEM,ARIMA,390.48,35.87,36.90,35.66,35.87
4,SCORPION,SCORPION,ARIMA,352.87,51.25,53.99,52.49,51.25
...,...,...,...,...,...,...,...,...
65,L 8073 UL (EX.L 8341 UR),L 8341 UR,ARIMA,36.58,8.95,8.61,8.73,8.95
66,PRENJAK,PRENJAK,ARIMA,35.66,24.81,33.03,26.96,24.81
67,L 8898 UQ,L 8898 UQ,Ensemble,35.34,128.48,228.29,160.95,160.95
68,MATAHARI,MATAHARI,ARIMA,35.30,13.03,34.63,18.02,13.03



=== Detail Prediksi vs Aktual HM Per Bulan ===


,EQUIP NAME,NAMA_MASTER_TERPETAKAN,Bulan,Aktual_HM,Prediksi_HM_Terpilih,Selisih_HM,Model_Terpilih_Unit,MAPE_Terpilih (%),RMSE_Terpilih
0,SAKURA,SAKURA,2025-01,78.0,63.25,-14.75,Gradient Boosting,1109.74,51.15
1,SAKURA,SAKURA,2025-02,71.0,63.25,-7.75,Gradient Boosting,1109.74,51.15
2,SAKURA,SAKURA,2025-03,57.0,63.25,6.25,Gradient Boosting,1109.74,51.15
3,SAKURA,SAKURA,2025-04,32.0,63.25,31.25,Gradient Boosting,1109.74,51.15
4,SAKURA,SAKURA,2025-05,16.0,63.25,47.25,Gradient Boosting,1109.74,51.15
...,...,...,...,...,...,...,...,...,...
835,KOLIBRI,KOLIBRI,2025-08,165.0,170.71,5.71,ARIMA,35.02,40.52
836,KOLIBRI,KOLIBRI,2025-09,141.0,170.58,29.58,ARIMA,35.02,40.52
837,KOLIBRI,KOLIBRI,2025-10,160.0,170.56,10.56,ARIMA,35.02,40.52
838,KOLIBRI,KOLIBRI,2025-11,180.0,170.58,-9.42,ARIMA,35.02,40.52


### F2: List Unit yang Diexclude beserta Alasan Masing-masing

In [5]:
df_excl_sorted = df_excl.sort_values(['Alasan', 'EQUIP NAME']).reset_index(drop=True)

print("=== ANALISA 2: Unit Dikecualikan dari Proses Forecast 2022-2025 ===")
print(f"Total unit dikecualikan: {len(df_excl_sorted)}\n")
print("Distribusi per Alasan:")
for alasan, count in df_excl['Alasan'].value_counts().items():
    print(f"  [{count:3d} unit] {alasan}")

print()
display(df_excl_sorted)

=== ANALISA 2: Unit Dikecualikan dari Proses Forecast 2022-2025 ===
Total unit dikecualikan: 66

Distribusi per Alasan:
  [ 38 unit] HM/LITER = 0 selama 1 tahun penuh di Data Latih (2024).
  [ 17 unit] HM/LITER = 0 selama 1 tahun penuh di Data Latih (2022).
  [  6 unit] HM/LITER = 0 selama 1 tahun penuh di Data Uji (2025).
  [  5 unit] HM/LITER = 0 selama 1 tahun penuh di Data Latih (2023).



,EQUIP NAME,NAMA_MASTER_TERPETAKAN,Alasan
0,AKASIA,AKASIA,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
1,CAMAR,CAMAR,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
2,FLAMINGO,FLAMINGO,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
3,JATAYU,JATAYU,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
4,JATI,JATI,HM/LITER = 0 selama 1 tahun penuh di Data Lati...
...,...,...,...
61,L 8085 SK ( EX L 7287 TF ),L 8085 SK ( EX L 7287 TF ),HM/LITER = 0 selama 1 tahun penuh di Data Uji ...
62,L 8660 UN (EX. L7918MU),L 8660 UN (EX. L7918MU),HM/LITER = 0 selama 1 tahun penuh di Data Uji ...
63,L 9064 UR,L 9064 UR,HM/LITER = 0 selama 1 tahun penuh di Data Uji ...
64,L 9118 UQ,L 9118 UQ,HM/LITER = 0 selama 1 tahun penuh di Data Uji ...


### F3: Analisa Unit Mana Saja yang HM Tidak Bertambah Tapi Melakukan Pengisian BBM

In [6]:
import re

# ============================================================
# LOAD NAMA UNIT DARI FILE MASTER (sama persis dengan forecast)
# ============================================================
def load_master_names():
    master_df    = pd.read_excel('cost & bbm 2022 sd 2025 HP & Type.xlsx', header=1)
    master_names = set(master_df['NAMA ALAT BERAT'].dropna().astype(str).str.strip())
    return master_names

def get_mapped_unit_name(unit_name, master_names):
    hardcoded = {
        "FL RENTAL 01":                "FL RENTAL 01 TIMIKA",
        "TOBATI (EX.FL KALMAR 32T)":   "TOP LOADER KALMAR 35T/TOBATI",
        "L 8477 UUC (EX.L 9902 UR)":   "L 9902 UR / S75",
        "WIND RIVER (EX.TL BOSS 42T)":  "TOP LOADER BOSS"
    }
    unit_name = str(unit_name).strip()
    if unit_name in hardcoded and hardcoded[unit_name] in master_names:
        return hardcoded[unit_name]
    if unit_name in master_names:
        return unit_name
    if " (" in unit_name:
        before_paren = unit_name.split(" (")[0].strip()
        if before_paren in master_names:
            return before_paren
    if "EX." in unit_name.upper():
        match_ex = re.search(r'EX\.([^\)]+)', unit_name.upper())
        if match_ex:
            after_ex = match_ex.group(1).strip()
            if after_ex in master_names:
                return after_ex
    return None

master_names = load_master_names()
print(f"Total nama unit di file master: {len(master_names)}")

# ============================================================
# LOAD DATA BBM HARIAN
# ============================================================
def load_and_melt_excel(file_path, target_sheets):
    print(f"Memuat: {file_path}")
    all_data = []
    xls = pd.read_excel(file_path, sheet_name=None, header=[0, 1, 2])
    for sheet_name, df in xls.items():
        if sheet_name not in target_sheets:
            continue
        try:
            df = df.set_index(df.columns[0])
            df.index.name = 'TANGGAL'
            df.columns = df.columns.droplevel(1)
            df_stacked = df.stack(level=0).reset_index()
            df_stacked.rename(columns={'level_1': 'EQUIP NAME'}, inplace=True)
            all_data.append(df_stacked)
        except Exception:
            continue
    if not all_data:
        return pd.DataFrame()
    df_final = pd.concat(all_data, ignore_index=True)
    df_final['TANGGAL'] = pd.to_datetime(df_final['TANGGAL'], format='%d-%m-%Y', errors='coerce')
    df_final = df_final.dropna(subset=['TANGGAL'])
    return df_final

sheets_2022     = ['JAN', 'FEB', 'MAR', 'APR', 'MEI', 'JUN', 'JUL', 'AGT', 'SEP', 'OKT', 'NOV', 'DES']
sheets_2023     = ['01. Jan', '02. Feb', '03. Mar', '04. Apr', '05. Mei', '06. Jun',
                   '07. Jul', '08. Aug', '09. Sep', '10. Okt', '11. Nov', '12. Des']
sheets_2024     = ['JANUARI 24', 'FEB 24', 'maret 24', 'April 24', 'Mei 24', 'Juni 24',
                   'Juli 24', 'agt 24', 'sept 24', 'okt 24', 'nov 24', 'des 24']
sheets_des_2025 = ['Des 25']
sheets_2025     = ['JAN', 'FEB', 'MAR', 'APR', 'MEI', 'JUN', 'JUL', 'AGT', 'SEP', 'OKT', 'NOV']

df_2022  = load_and_melt_excel('BBM AAB 2022.xlsx',            target_sheets=sheets_2022)
df_2023  = load_and_melt_excel('BBM AAB 2023.xlsx',            target_sheets=sheets_2023)
df_2024  = load_and_melt_excel('BBM AAB 2024 & Des 2025.xlsx', target_sheets=sheets_2024)
df_des25 = load_and_melt_excel('BBM AAB 2024 & Des 2025.xlsx', target_sheets=sheets_des_2025)
df_2025  = load_and_melt_excel('BBM AAB Jan-Nov 2025.xlsx',    target_sheets=sheets_2025)

df_all = pd.concat([df_2022, df_2023, df_2024, df_des25, df_2025], ignore_index=True)
df_all = df_all.sort_values(['EQUIP NAME', 'TANGGAL']).reset_index(drop=True)

# ============================================================
# FILTER: HANYA UNIT YANG TERDAFTAR DI FILE MASTER
# ============================================================
df_all['NAMA_MASTER'] = df_all['EQUIP NAME'].apply(
    lambda x: get_mapped_unit_name(x, master_names)
)
df_all = df_all[df_all['NAMA_MASTER'].notna()].copy()

df_all['HM_raw']    = pd.to_numeric(df_all['HM'],    errors='coerce')
df_all['LITER_raw'] = pd.to_numeric(df_all['LITER'], errors='coerce').fillna(0)

print(f"\nSetelah filter unit master:")
print(f"  Total baris : {len(df_all):,}")
print(f"  Total unit  : {df_all['EQUIP NAME'].nunique()}")
print(f"  Rentang     : {df_all['TANGGAL'].min().date()} s.d. {df_all['TANGGAL'].max().date()}")

Total nama unit di file master: 245
Memuat: BBM AAB 2022.xlsx
Memuat: BBM AAB 2023.xlsx
Memuat: BBM AAB 2024 & Des 2025.xlsx
Memuat: BBM AAB 2024 & Des 2025.xlsx
Memuat: BBM AAB Jan-Nov 2025.xlsx

Setelah filter unit master:
  Total baris : 349,179
  Total unit  : 239
  Rentang     : 2022-01-01 s.d. 2025-12-31


In [7]:
# ============================================================
# AGREGASI KE BULANAN
# ============================================================
df_all['HM_Clean'] = df_all['HM_raw'].replace(0, np.nan)
df_all['HM_Clean'] = df_all.groupby('EQUIP NAME')['HM_Clean'].ffill().fillna(0)
df_all['Delta_HM'] = df_all.groupby('EQUIP NAME')['HM_Clean'].diff().fillna(0)
df_all.loc[df_all['Delta_HM'] < 0,  'Delta_HM'] = 0
df_all.loc[df_all['Delta_HM'] > 24, 'Delta_HM'] = 0

df_all['TAHUN_BULAN'] = df_all['TANGGAL'].dt.to_period('M')

df_monthly = df_all.groupby(['EQUIP NAME', 'NAMA_MASTER', 'TAHUN_BULAN']).agg(
    Delta_HM_Bulan = ('Delta_HM',  'sum'),
    LITER_Bulan    = ('LITER_raw', 'sum'),
    Jumlah_Isi_BBM = ('LITER_raw', lambda x: (x > 0).sum())
).reset_index()

# ============================================================
# TIPE A: HM=0 & LITER>0 minimal 3 bulan BERTURUT-TURUT
# ============================================================
THRESHOLD_STREAK = 3
anomali_a = []

for unit, grp in df_monthly.groupby('EQUIP NAME'):
    grp = grp.sort_values('TAHUN_BULAN').reset_index(drop=True)
    nama_master = grp['NAMA_MASTER'].iloc[0]

    grp['Janggal'] = (grp['Delta_HM_Bulan'] == 0) & (grp['LITER_Bulan'] > 0)

    streak       = 0
    streak_start = None
    streak_rows  = []

    for idx, row in grp.iterrows():
        if row['Janggal']:
            streak += 1
            if streak == 1:
                streak_start = row['TAHUN_BULAN']
            streak_rows.append(row)
        else:
            if streak >= THRESHOLD_STREAK:
                total_liter = sum(r['LITER_Bulan'] for r in streak_rows)
                total_isi   = sum(r['Jumlah_Isi_BBM'] for r in streak_rows)
                anomali_a.append({
                    'EQUIP NAME':             unit,
                    'NAMA_MASTER_TERPETAKAN': nama_master,
                    'Tipe_Anomali':           'Tipe A — Streak ≥ 3 Bulan',
                    'Bulan_Mulai':            str(streak_start),
                    'Bulan_Selesai':          str(streak_rows[-1]['TAHUN_BULAN']),
                    'Durasi_Bulan':           streak,
                    'Total_LITER_Terisi':     round(total_liter, 2),
                    'Total_Kali_Isi_BBM':     int(total_isi),
                    'Rata_LITER_per_Bulan':   round(total_liter / streak, 2),
                    'Keterangan':             f'HM=0 & LITER>0 selama {streak} bulan berturut-turut'
                })
            streak       = 0
            streak_start = None
            streak_rows  = []

    # Flush di akhir loop
    if streak >= THRESHOLD_STREAK:
        total_liter = sum(r['LITER_Bulan'] for r in streak_rows)
        total_isi   = sum(r['Jumlah_Isi_BBM'] for r in streak_rows)
        anomali_a.append({
            'EQUIP NAME':             unit,
            'NAMA_MASTER_TERPETAKAN': nama_master,
            'Tipe_Anomali':           'Tipe A — Streak ≥ 3 Bulan',
            'Bulan_Mulai':            str(streak_start),
            'Bulan_Selesai':          str(streak_rows[-1]['TAHUN_BULAN']),
            'Durasi_Bulan':           streak,
            'Total_LITER_Terisi':     round(total_liter, 2),
            'Total_Kali_Isi_BBM':     int(total_isi),
            'Rata_LITER_per_Bulan':   round(total_liter / streak, 2),
            'Keterangan':             f'HM=0 & LITER>0 selama {streak} bulan berturut-turut'
        })

# ============================================================
# TIPE B: Pengisian BBM terisolir di tengah periode tidak aktif
# Syarat: 2 bulan sebelum HM=0, bulan ini HM=0 & LITER>0,
#         2 bulan sesudah HM=0
# ============================================================
JENDELA = 2
anomali_b = []

# Kumpulkan semua bulan yang sudah masuk Tipe A agar tidak double count
tipe_a_keys = set()
for rec in anomali_a:
    unit  = rec['EQUIP NAME']
    start = pd.Period(rec['Bulan_Mulai'], freq='M')
    end   = pd.Period(rec['Bulan_Selesai'], freq='M')
    p = start
    while p <= end:
        tipe_a_keys.add((unit, str(p)))
        p += 1

for unit, grp in df_monthly.groupby('EQUIP NAME'):
    grp = grp.sort_values('TAHUN_BULAN').reset_index(drop=True)
    nama_master = grp['NAMA_MASTER'].iloc[0]
    n = len(grp)

    for idx in range(n):
        row = grp.iloc[idx]

        # Kandidat: bulan ini HM=0 tapi LITER>0
        if not (row['Delta_HM_Bulan'] == 0 and row['LITER_Bulan'] > 0):
            continue

        # Skip jika bulan ini sudah masuk Tipe A
        if (unit, str(row['TAHUN_BULAN'])) in tipe_a_keys:
            continue

        # Cek 2 bulan SEBELUM — semua harus HM=0
        before_ok = True
        for j in range(1, JENDELA + 1):
            prev_idx = idx - j
            if prev_idx < 0:
                # Tidak ada data sebelumnya — anggap tidak memenuhi syarat
                before_ok = False
                break
            if grp.iloc[prev_idx]['Delta_HM_Bulan'] != 0:
                before_ok = False
                break

        if not before_ok:
            continue

        # Cek 2 bulan SESUDAH — semua harus HM=0
        # Untuk bulan sesudah yang merupakan pengisian BBM lagi (HM=0, LITER>0)
        # tetap dianggap memenuhi syarat karena konteksnya sama-sama tidak aktif
        after_ok = True
        for j in range(1, JENDELA + 1):
            next_idx = idx + j
            if next_idx >= n:
                # Tidak ada data sesudahnya — anggap tidak memenuhi syarat
                after_ok = False
                break
            if grp.iloc[next_idx]['Delta_HM_Bulan'] != 0:
                after_ok = False
                break

        if not after_ok:
            continue

        # Kumpulkan konteks 2 bulan sebelum dan sesudah untuk keterangan
        before_periods = [str(grp.iloc[idx - j]['TAHUN_BULAN'])
                          for j in range(JENDELA, 0, -1) if idx - j >= 0]
        after_periods  = [str(grp.iloc[idx + j]['TAHUN_BULAN'])
                          for j in range(1, JENDELA + 1) if idx + j < n]

        anomali_b.append({
            'EQUIP NAME':             unit,
            'NAMA_MASTER_TERPETAKAN': nama_master,
            'Tipe_Anomali':           'Tipe B — Pengisian Terisolir',
            'Bulan_Mulai':            str(row['TAHUN_BULAN']),
            'Bulan_Selesai':          str(row['TAHUN_BULAN']),
            'Durasi_Bulan':           1,
            'Total_LITER_Terisi':     round(row['LITER_Bulan'], 2),
            'Total_Kali_Isi_BBM':     int(row['Jumlah_Isi_BBM']),
            'Rata_LITER_per_Bulan':   round(row['LITER_Bulan'], 2),
            'Keterangan':             (
                f"HM=0 & LITER>0 di bulan ini. "
                f"2 bulan sebelum ({', '.join(before_periods)}) HM=0. "
                f"2 bulan sesudah ({', '.join(after_periods)}) HM=0."
            )
        })

# ============================================================
# GABUNGKAN TIPE A DAN TIPE B
# ============================================================
df_anomali_a = pd.DataFrame(anomali_a)
df_anomali_b = pd.DataFrame(anomali_b)

df_anomali = pd.concat([df_anomali_a, df_anomali_b], ignore_index=True)
df_anomali = df_anomali.sort_values(
    ['EQUIP NAME', 'Bulan_Mulai']
).reset_index(drop=True)

# ============================================================
# SUMMARY
# ============================================================
print(f"=== ANALISA 3: Deteksi Anomali HM Tetap tapi Isi BBM ===")
print(f"Hanya unit yang terdaftar di file master\n")
print(f"TIPE A (Streak ≥ {THRESHOLD_STREAK} bulan berturut-turut):")
print(f"  Kejadian : {len(df_anomali_a)}")
print(f"  Unit unik: {df_anomali_a['EQUIP NAME'].nunique() if len(df_anomali_a) > 0 else 0}")
print(f"\nTIPE B (Pengisian terisolir, jendela {JENDELA} bulan sebelum & sesudah):")
print(f"  Kejadian : {len(df_anomali_b)}")
print(f"  Unit unik: {df_anomali_b['EQUIP NAME'].nunique() if len(df_anomali_b) > 0 else 0}")
print(f"\nTotal gabungan:")
print(f"  Kejadian : {len(df_anomali)}")
print(f"  Unit unik: {df_anomali['EQUIP NAME'].nunique() if len(df_anomali) > 0 else 0}")

from IPython.display import display
print("\n--- Tipe A ---")
display(df_anomali_a.sort_values(['Durasi_Bulan', 'Total_LITER_Terisi'],
                                  ascending=[False, False]).reset_index(drop=True))
print("\n--- Tipe B ---")
display(df_anomali_b.sort_values('Total_LITER_Terisi',
                                  ascending=False).reset_index(drop=True))

=== ANALISA 3: Deteksi Anomali HM Tetap tapi Isi BBM ===
Hanya unit yang terdaftar di file master

TIPE A (Streak ≥ 3 bulan berturut-turut):
  Kejadian : 7
  Unit unik: 6

TIPE B (Pengisian terisolir, jendela 2 bulan sebelum & sesudah):
  Kejadian : 15
  Unit unik: 7

Total gabungan:
  Kejadian : 22
  Unit unik: 12

--- Tipe A ---


,EQUIP NAME,NAMA_MASTER_TERPETAKAN,Tipe_Anomali,Bulan_Mulai,Bulan_Selesai,Durasi_Bulan,Total_LITER_Terisi,Total_Kali_Isi_BBM,Rata_LITER_per_Bulan,Keterangan
0,SANY KUPP CAB TUAL,SANY KUPP CAB TUAL,Tipe A — Streak ≥ 3 Bulan,2023-01,2024-06,18,5825.00,56,323.61,HM=0 & LITER>0 selama 18 bulan berturut-turut
1,SIDE LOUDER BOSS,SIDE LOUDER BOSS,Tipe A — Streak ≥ 3 Bulan,2022-01,2023-02,14,24083.00,251,1720.21,HM=0 & LITER>0 selama 14 bulan berturut-turut
2,SIDE LOUDER BOSS,SIDE LOUDER BOSS,Tipe A — Streak ≥ 3 Bulan,2023-04,2024-04,13,18765.00,162,1443.46,HM=0 & LITER>0 selama 13 bulan berturut-turut
3,L 8999 UUC (EX.L 9703 UQ),L 9703 UQ,Tipe A — Streak ≥ 3 Bulan,2023-02,2023-04,3,5712.05,30,1904.02,HM=0 & LITER>0 selama 3 bulan berturut-turut
4,KENARI,KENARI,Tipe A — Streak ≥ 3 Bulan,2022-01,2022-03,3,470.00,17,156.67,HM=0 & LITER>0 selama 3 bulan berturut-turut
5,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe A — Streak ≥ 3 Bulan,2023-09,2023-11,3,395.00,4,131.67,HM=0 & LITER>0 selama 3 bulan berturut-turut
6,KUDA PUTIH,KUDA PUTIH,Tipe A — Streak ≥ 3 Bulan,2022-10,2022-12,3,90.00,3,30.00,HM=0 & LITER>0 selama 3 bulan berturut-turut



--- Tipe B ---


,EQUIP NAME,NAMA_MASTER_TERPETAKAN,Tipe_Anomali,Bulan_Mulai,Bulan_Selesai,Durasi_Bulan,Total_LITER_Terisi,Total_Kali_Isi_BBM,Rata_LITER_per_Bulan,Keterangan
0,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2024-07,2024-07,1,200.00,1,200.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
1,TITAN,TITAN,Tipe B — Pengisian Terisolir,2024-03,2024-03,1,200.00,1,200.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
2,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2024-10,2024-10,1,160.00,1,160.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
3,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2022-09,2022-09,1,150.00,1,150.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
4,AKASIA,AKASIA,Tipe B — Pengisian Terisolir,2022-05,2022-05,1,100.00,1,100.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
5,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2023-05,2023-05,1,100.00,1,100.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
6,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2025-03,2025-03,1,89.98,1,89.98,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
7,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2024-01,2024-01,1,80.00,1,80.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
8,SIDE LOADER BOSS 3,SIDE LOADER BOSS 3,Tipe B — Pengisian Terisolir,2023-06,2023-06,1,70.00,1,70.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...
9,TERATAI,TERATAI,Tipe B — Pengisian Terisolir,2023-11,2023-11,1,50.00,1,50.00,HM=0 & LITER>0 di bulan ini. 2 bulan sebelum (...


In [ ]:
OUTPUT_FILE = 'Hasil_Analisa_Lanjutan.xlsx'

unit_anomali       = df_anomali['EQUIP NAME'].unique() if len(df_anomali) > 0 else []
df_monthly_anomali = df_monthly[
    df_monthly['EQUIP NAME'].isin(unit_anomali)
].copy()
df_monthly_anomali['TAHUN_BULAN'] = df_monthly_anomali['TAHUN_BULAN'].astype(str)
df_monthly_anomali = df_monthly_anomali.sort_values(['EQUIP NAME', 'TAHUN_BULAN'])

df_export_a1_detail = df_detail_tampil[[
    'EQUIP NAME', 'NAMA_MASTER_TERPETAKAN', 'Bulan',
    'Aktual_HM', 'Prediksi_HM_Terpilih', 'Selisih_HM',
    'Model_Terpilih_Unit', 'MAPE_Terpilih (%)', 'RMSE_Terpilih',
    'RMSE_HM_ARIMA', 'RMSE_HM_GB', 'RMSE_HM_Ensemble'
]].round(2)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    df_ringkasan_over35.round(2).to_excel(
        writer, sheet_name='A1_Ringkasan_MAPE_Over35',     index=False)
    df_export_a1_detail.to_excel(
        writer, sheet_name='A1_Detail_Prediksi_vs_Aktual', index=False)
    df_excl_sorted.to_excel(
        writer, sheet_name='A2_Unit_Dikecualikan',          index=False)

    if len(df_anomali) > 0:
        # Sheet gabungan semua anomali
        df_anomali.to_excel(
            writer, sheet_name='A3_Semua_Anomali',          index=False)
        # Sheet terpisah per tipe
        if len(df_anomali_a) > 0:
            df_anomali_a.sort_values(
                ['Durasi_Bulan', 'Total_LITER_Terisi'],
                ascending=[False, False]
            ).reset_index(drop=True).to_excel(
                writer, sheet_name='A3_TipeA_Streak_3Bulan', index=False)
        if len(df_anomali_b) > 0:
            df_anomali_b.sort_values(
                'Total_LITER_Terisi', ascending=False
            ).reset_index(drop=True).to_excel(
                writer, sheet_name='A3_TipeB_Terisolir',     index=False)
        # Detail bulanan untuk investigasi
        df_monthly_anomali.to_excel(
            writer, sheet_name='A3_Detail_Bulanan_Anomali',  index=False)
    else:
        pd.DataFrame({'Info': ['Tidak ditemukan anomali']}).to_excel(
            writer, sheet_name='A3_Semua_Anomali',           index=False)

print(f"Seluruh hasil berhasil disimpan ke '{OUTPUT_FILE}'")
print(f"\nSheet yang dihasilkan:")
print(f"  A1_Ringkasan_MAPE_Over35       — {len(df_ringkasan_over35)} unit")
print(f"  A1_Detail_Prediksi_vs_Aktual   — {len(df_export_a1_detail)} baris")
print(f"  A2_Unit_Dikecualikan           — {len(df_excl_sorted)} unit")
print(f"  A3_Semua_Anomali               — {len(df_anomali)} kejadian")
print(f"  A3_TipeA_Streak_3Bulan         — {len(df_anomali_a)} kejadian")
print(f"  A3_TipeB_Terisolir             — {len(df_anomali_b)} kejadian")
print(f"  A3_Detail_Bulanan_Anomali      — {len(df_monthly_anomali)} baris")

Seluruh hasil berhasil disimpan ke 'Hasil_Analisa_Lanjutan.xlsx'

Sheet yang dihasilkan:
  A1_Ringkasan_MAPE_Over35       — 70 unit
  A1_Detail_Prediksi_vs_Aktual   — 840 baris
  A2_Unit_Dikecualikan           — 66 unit
  A3_Semua_Anomali               — 22 kejadian
  A3_TipeA_Streak_3Bulan         — 7 kejadian
  A3_TipeB_Terisolir             — 15 kejadian
  A3_Detail_Bulanan_Anomali      — 576 baris
